# CellFlow implementation
To run this analysis, we created a new conda environment (specified in analysis/scripts/cellflow_env.yaml), and followed this tutorial to obtain esm2 embedding: https://github.com/theislab/CellFlow/blob/docs/tutorial/satija/docs/notebooks/300_satija_unseen_cell_line_cytokine.ipynb

# Get ESM2 embeddings

In [2]:
import scanpy as sc

In [ ]:
import os
def get_esm_embedding_adata(adata, adata_path_esm):
    from cellflow.preprocessing import get_esm_embedding
    get_esm_embedding(adata, 
                    gene_key="gene_ensembl", 
                    gene_emb_key="esm_embeddings", 
                    null_value = "control", 
                    esm_model_name="esm2_t36_3B_UR50D",
                    use_cuda=True,
                    cache_dir='./data/esm_cache')

    for key in adata.uns['esm_embeddings'].keys():
        adata.uns['esm_embeddings'][key] = adata.uns['esm_embeddings'][key].cpu().detach().numpy()
    
    if 'esm_embeddings_metadata' in adata.uns:
        meta = adata.uns['esm_embeddings_metadata']
        for col in meta.columns:
            if meta[col].dtype == object:
                meta[col] = meta[col].fillna('').astype(str)
            elif meta[col].dtype == bool:
                meta[col] = meta[col].astype(int)
            else:
                meta[col] = meta[col].fillna(-1)
        adata.uns['esm_embeddings_metadata'] = meta
    adata.write_h5ad(adata_path_esm)

## Replogle K562

In [ ]:
adata = sc.read_h5ad("../data/preprocessed_replogle_k562.h5ad")

In [ ]:
adata.obs["is_control"] = adata.obs.apply(
    lambda x: True if x["condition"] == "ctrl" else False, axis=1
)

In [ ]:
perturbation = []
for cond in adata.obs['condition']:
    if cond == 'ctrl':
        perturbation.append('control')
    else:
        perturbation.append(cond.split('+')[0])
adata.obs['perturbation_name'] = perturbation

In [ ]:
from mygene import MyGeneInfo

mg = MyGeneInfo()
genes = adata.obs['perturbation_name'].unique().tolist()
out = mg.querymany(genes, scopes='symbol', fields='ensembl.gene', species='human', returnall=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
29 input query terms found no hit:	['N6AMT1', 'control', 'CENPJ', 'C7orf26', 'GARS', 'WDR61', 'HARS', 'RARS', 'TARS', 'QARS', 'PHB', 'L


In [ ]:
out['missing']

['N6AMT1',
 'control',
 'CENPJ',
 'C7orf26',
 'GARS',
 'WDR61',
 'HARS',
 'RARS',
 'TARS',
 'QARS',
 'PHB',
 'LARS',
 'NARS',
 'PRPF4B',
 'C14orf178',
 'CD3EAP',
 'VARS',
 'MARS',
 'NEPRO',
 'ZNRD1',
 'TWISTNB',
 'EPRS',
 'KARS',
 'C12orf45',
 'DARS',
 'CCDC84',
 'AARS',
 'IARS',
 'SARS']

In [ ]:
gene_name_map = {
    'N6AMT1': 'HEMK2',
    'control': 'control',
    'CENPJ': 'CPAP',
    'C7orf26': 'INTS15',
    'GARS': 'GARS1',
    'WDR61': 'SKIC8',
    'HARS': 'HARS1',
    'RARS': 'RARS1',
    'TARS': 'TARS1',
    'QARS': 'QARS1',
    'PHB': 'PHB1',
    'LARS': 'LARS1',
    'NARS': 'NARS1',
    'PRPF4B': 'PRP4K',
    'C14orf178': 'SPTSSA',
    'CD3EAP': 'POLR1G',
    'VARS': 'VARS1',
    'MARS': 'MARS1',
    'NEPRO': 'RMP64',
    'ZNRD1': 'POLR1H',
    'TWISTNB': 'POLR1F',
    'EPRS': 'EPRS1',
    'KARS': 'KARS1',
    'C12orf45': 'NOPCHAP1',
    'DARS': 'DARS1',
    'CCDC84': 'CENATAC',
    'AARS': 'AARS1',
    'IARS': 'IARS1',
    'SARS': 'SARS1'
}
additional_genes = [gene_name_map[g] for g in gene_name_map.keys()]
out_2 = mg.querymany(additional_genes, scopes='symbol', fields='ensembl.gene', species='human', returnall=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found no hit:	['control']


In [ ]:
out['out'].extend(out_2['out'])

In [ ]:
gene_name_old = list(gene_name_map.keys())
gene_to_ensembl = {}
for gene in out['out']:
    if gene["query"] in additional_genes:
        old_gene_name = [k for k,v in gene_name_map.items() if v == gene["query"]][0]
        if old_gene_name == 'control':
            gene_to_ensembl[old_gene_name] = 'control'
        else:
            if len(gene["ensembl"]) > 1:
                gene_to_ensembl[old_gene_name] = gene["ensembl"][0]["gene"] # take the first one
            else:
                gene_to_ensembl[old_gene_name] = gene["ensembl"]["gene"]
    elif gene["query"] in gene_name_old:
        continue
    elif len(gene["ensembl"]) > 1:
        gene_to_ensembl[gene["query"]] = gene["ensembl"][0]["gene"] # take the first one
    else:
        gene_to_ensembl[gene["query"]] = gene["ensembl"]["gene"]        

In [ ]:
gene_to_ensembl['control'] = 'control'

adata.obs["gene_ensembl"] = adata.obs["perturbation_name"].map(gene_to_ensembl)

In [ ]:
# randomly select 5000 cells from controls
import numpy as np
adata_ctrl = adata[adata.obs["is_control"], :].copy()
np.random.seed(0)
selected_control_indices = np.random.choice(adata_ctrl.obs_names, size=5000, replace=False)
selected_control = adata_ctrl[selected_control_indices, :]

adata_no_ctrl = adata[~adata.obs["is_control"], :].copy()
adata = sc.concat([adata_no_ctrl, selected_control], axis=0)

In [ ]:
adata_path_esm = "../data/preprocessed_replogle_k562_with_esm.h5ad"
get_esm_embedding_adata(adata, adata_path_esm)

## Replogle RPE1

In [ ]:
adata = sc.read_h5ad("../data/preprocessed_replogle_rpe1.h5ad")

In [ ]:
adata

AnnData object with n_obs × n_vars = 162734 × 5788
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells', 'gene_name', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'neighbors', 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'pca', 'rank_genes_groups_cov_all', 'rank_genes_groups_list', 'top_non_dropout_de_20', 'top_non_zero_de_20', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [ ]:
adata.obs["is_control"] = adata.obs.apply(
    lambda x: True if x["condition"] == "ctrl" else False, axis=1
)

In [ ]:
perturbation = []
for cond in adata.obs['condition']:
    if cond == 'ctrl':
        perturbation.append('control')
    else:
        perturbation.append(cond.split('+')[0])
adata.obs['perturbation_name'] = perturbation

In [ ]:
from mygene import MyGeneInfo

mg = MyGeneInfo()
genes = adata.obs['perturbation_name'].unique().tolist()
out = mg.querymany(genes, scopes='symbol', fields='ensembl.gene', species='human', returnall=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found dup hits:	[('CAST', 2)]
38 input query terms found no hit:	['control', 'CCDC130', 'KARS', 'C14orf178', 'C7orf26', 'NARS', 'VARS', 'HARS', 'QARS', 'CCDC84', 'PH


In [ ]:
out['missing']

['control',
 'CCDC130',
 'KARS',
 'C14orf178',
 'C7orf26',
 'NARS',
 'VARS',
 'HARS',
 'QARS',
 'CCDC84',
 'PHB',
 'ALG1L',
 'GARS',
 'C9orf16',
 'MARS',
 'TARS',
 'C5orf30',
 'RARS',
 'FAM102B',
 'H2AFZ',
 'WARS',
 'DARS',
 'YARS',
 'SPATA5L1',
 'CCDC115',
 'WDR61',
 'LARS',
 'CARS',
 'AC118549.1',
 'CENPJ',
 'AARS',
 'PRPF4B',
 'SPATA5',
 'EPRS',
 'IARS',
 'CD3EAP',
 'ZNRD1',
 'ZNF720']

In [ ]:
gene_name_map = {
    'control': 'control',
    'CCDC130': 'YJU2B',
    'KARS': 'KARS1',
    'C14orf178': 'SPTSSA',
    'C7orf26': 'INTS15',
    'NARS': 'NARS1',
    'VARS': 'VARS1',
    'HARS': 'HARS1',
    'QARS': 'QARS1',
    'CCDC84': 'CENATAC',
    'PHB': 'PHB1',
    'ALG1L': 'ALG1L1P',
    'GARS': 'GARS1',
    'C9orf16': 'BBLN',
    'MARS': 'MARS1',
    'TARS': 'TARS1',
    'C5orf30': 'MACIR',
    'RARS': 'RARS1',
    'FAM102B': 'EEIG2',
    'H2AFZ': 'H2AZ1',
    'WARS': 'WARS1',
    'DARS': 'DARS1',
    'YARS': 'YARS1',
    'SPATA5L1': 'AFG2B',
    'CCDC115': 'VMA22',
    'WDR61': 'SKIC8',
    'LARS': 'LARS1',
    'CARS': 'CARS1',
    'AC118549.1': 'ZZZ3',
    'CENPJ': 'CPAP',
    'AARS': 'AARS1',
    'PRPF4B': 'PRP4K',
    'SPATA5': 'AFG2A',
    'EPRS': 'EPRS1',
    'IARS': 'IARS1',
    'CD3EAP': 'POLR1G',
    'ZNRD1': 'POLR1H',
    'ZNF720': 'KRBOX5'
}
additional_genes = [gene_name_map[g] for g in gene_name_map.keys()]
out_2 = mg.querymany(additional_genes, scopes='symbol', fields='ensembl.gene', species='human', returnall=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found dup hits:	[('ALG1L1P', 2)]
2 input query terms found no hit:	['control', 'KRBOX5']


In [ ]:
out['out'].extend(out_2['out'])

In [ ]:
gene_name_old = list(gene_name_map.keys())
for gene in out['out']:
    try:
        if gene['query'] in gene_name_old:
            continue
        if gene['ensembl']:
            continue
    except KeyError:
        print(gene)


{'query': 'ALG1L1P', '_id': '200810', '_score': 8.24828}
{'query': 'KRBOX5', 'notfound': True}


In [ ]:
gene_name_old = list(gene_name_map.keys())
gene_to_ensembl = {}
for gene in out['out']:
    if gene["query"] == 'control':
        gene_to_ensembl['control'] = 'control'
        continue
    if gene["query"] in additional_genes:
        old_gene_name = [k for k,v in gene_name_map.items() if v == gene["query"]][0]
    elif gene["query"] in gene_name_old:
        continue
    else:
        old_gene_name = gene["query"]
    try:
        if len(gene["ensembl"]) > 1:
            gene_to_ensembl[old_gene_name] = gene["ensembl"][0]["gene"] # take the first one
        else:
            gene_to_ensembl[old_gene_name] = gene["ensembl"]["gene"]
    except (KeyError, IndexError):
        print(f"Error processing gene: {gene}")

Error processing gene: {'query': 'ALG1L1P', '_id': '200810', '_score': 8.24828}
Error processing gene: {'query': 'KRBOX5', 'notfound': True}


In [ ]:
gene_to_ensembl['control'] = 'control'
gene_to_ensembl['ALG1L1P'] = 'ENSG00000189366'
gene_to_ensembl['KRBOX5'] = 'ENSG00000197302'
gene_to_ensembl['ZNF720'] = 'ENSG00000197302'

adata.obs["gene_ensembl"] = adata.obs["perturbation_name"].map(gene_to_ensembl)

In [ ]:
# randomly select 5000 cells from controls
import numpy as np
adata_ctrl = adata[adata.obs["is_control"], :].copy()
np.random.seed(0)
selected_control_indices = np.random.choice(adata_ctrl.obs_names, size=5000, replace=False)
selected_control = adata_ctrl[selected_control_indices, :]

adata_no_ctrl = adata[~adata.obs["is_control"], :].copy()
adata = sc.concat([adata_no_ctrl, selected_control], axis=0)

In [ ]:
adata_path_esm = "../data/preprocessed_replogle_rpe1_with_esm.h5ad"
get_esm_embedding_adata(adata, adata_path_esm)

## Nadig hepg2

In [ ]:
adata_hepg2 = sc.read_h5ad('../data/processed_nadig_hepg2.h5ad')
adata_hepg2_orig = sc.read_h5ad('../data/GSE264667_hepg2_raw_singlecell_01_filtered.h5ad', backed='r')

In [ ]:
adata_hepg2.obs["is_control"] = adata_hepg2.obs.apply(
    lambda x: True if x["condition"] == "ctrl" else False, axis=1
)

In [ ]:
from mygene import MyGeneInfo

mg = MyGeneInfo()
genes = adata_hepg2_orig.obs['gene_id'].unique().tolist()
out = mg.querymany(genes, scopes='ensembl.gene', fields='ensembl.gene', species='human', returnall=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
3 input query terms found no hit:	['non-targeting', 'ENSG00000112096', 'ENSG00000277203']


In [ ]:
ensembl_map = {
    'ENSG00000112096': 'ENSG00000291237',
    'ENSG00000277203': 'ENSG00000288722',
    'non-targeting': 'control'
}
additional_genes = [ensembl_map[g] for g in ensembl_map.keys()]
out_2 = mg.querymany(additional_genes, scopes='ensembl.gene', fields='ensembl.gene', species='human', returnall=True)
out['out'].extend(out_2['out'])

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found no hit:	['control']


In [ ]:
gene_name_old = list(ensembl_map.keys())
gene_to_ensembl = {}
for gene in out['out']:
    if gene["query"] in additional_genes:
        old_gene_name = [k for k,v in ensembl_map.items() if v == gene["query"]][0]
        if old_gene_name == 'non-targeting':
            gene_to_ensembl[old_gene_name] = 'control'
        else:
            if len(gene["ensembl"]) > 1:
                gene_to_ensembl[old_gene_name] = gene["ensembl"][0]["gene"] # take the first one
            else:
                gene_to_ensembl[old_gene_name] = gene["ensembl"]["gene"]
    elif (gene["query"] in gene_name_old) or (gene["query"] == 'non-targeting'):
        continue
    elif len(gene["ensembl"]) > 1:
        gene_to_ensembl[gene["query"]] = gene["ensembl"][0]["gene"] # take the first one
    else:
        gene_to_ensembl[gene["query"]] = gene["ensembl"]["gene"]  

In [ ]:
adata_hepg2.obs['gene_ensembl'] = adata_hepg2_orig.obs['gene_id'].map(gene_to_ensembl)

In [ ]:
adata_path_esm = "../data/preprocessed_nadig_hepg2_with_esm.h5ad"
get_esm_embedding_adata(adata_hepg2, adata_path_esm)

## Nadig jurkat

In [ ]:
adata_jurkat = sc.read_h5ad('../data/processed_nadig_jurkat.h5ad')
adata_jurkat_orig = sc.read_h5ad('../data/GSE264667_jurkat_raw_singlecell_01_filtered.h5ad', backed='r')

In [ ]:
adata_jurkat.obs["is_control"] = adata_jurkat.obs.apply(
    lambda x: True if x["condition"] == "ctrl" else False, axis=1
)

In [ ]:
# remove 'ENSG00000153113' in adata_jurkat, not available
adata_jurkat = adata_jurkat[adata_jurkat.obs['gene_ensembl'] != 'ENSG00000153113']

In [ ]:
from mygene import MyGeneInfo

mg = MyGeneInfo()
genes = adata_jurkat_orig.obs['gene_id'].unique().tolist()
out = mg.querymany(genes, scopes='ensembl.gene', fields='ensembl.gene', species='human', returnall=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
3 input query terms found no hit:	['non-targeting', 'ENSG00000277203', 'ENSG00000112096']


In [ ]:
ensembl_map = {
    'ENSG00000112096': 'ENSG00000291237',
    'ENSG00000277203': 'ENSG00000288722',
    'non-targeting': 'control'
}
additional_genes = [ensembl_map[g] for g in ensembl_map.keys()]
out_2 = mg.querymany(additional_genes, scopes='ensembl.gene', fields='ensembl.gene', species='human', returnall=True)
out['out'].extend(out_2['out'])

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found no hit:	['control']


In [ ]:
gene_name_old = list(ensembl_map.keys())
gene_to_ensembl = {}
for gene in out['out']:
    if gene["query"] in additional_genes:
        old_gene_name = [k for k,v in ensembl_map.items() if v == gene["query"]][0]
        if old_gene_name == 'non-targeting':
            gene_to_ensembl[old_gene_name] = 'control'
        else:
            if len(gene["ensembl"]) > 1:
                gene_to_ensembl[old_gene_name] = gene["ensembl"][0]["gene"] # take the first one
            else:
                gene_to_ensembl[old_gene_name] = gene["ensembl"]["gene"]
    elif (gene["query"] in gene_name_old) or (gene["query"] == 'non-targeting'):
        continue
    elif len(gene["ensembl"]) > 1:
        gene_to_ensembl[gene["query"]] = gene["ensembl"][0]["gene"] # take the first one
    else:
        gene_to_ensembl[gene["query"]] = gene["ensembl"]["gene"]  

In [ ]:
adata_jurkat.obs['gene_ensembl'] = adata_jurkat_orig.obs['gene_id'].map(gene_to_ensembl)

In [ ]:
adata_path_esm = "../data/preprocessed_nadig_jurkat_with_esm.h5ad"
get_esm_embedding_adata(adata_jurkat, adata_path_esm)

## Norman combo

In [10]:
import os
def get_esm_embedding_adata(adata, adata_path_esm):
    from cellflow.preprocessing import get_esm_embedding
    get_esm_embedding(adata, 
                    gene_key=["gene_ensembl_1", "gene_ensembl_2"], 
                    gene_emb_key="esm_embeddings", 
                    null_value = "control", 
                    esm_model_name="esm2_t36_3B_UR50D",
                    use_cuda=True,
                    cache_dir='./data/esm_cache')

    for key in adata.uns['esm_embeddings'].keys():
        adata.uns['esm_embeddings'][key] = adata.uns['esm_embeddings'][key].cpu().detach().numpy()
    
    if 'esm_embeddings_metadata' in adata.uns:
        meta = adata.uns['esm_embeddings_metadata']
        for col in meta.columns:
            if meta[col].dtype == object:
                meta[col] = meta[col].fillna('').astype(str)
            elif meta[col].dtype == bool:
                meta[col] = meta[col].astype(int)
            else:
                meta[col] = meta[col].fillna(-1)
        adata.uns['esm_embeddings_metadata'] = meta
    adata.write_h5ad(adata_path_esm)

In [ ]:
adata = sc.read_h5ad("../data/preprocessed_norman_combo.h5ad")

In [ ]:
adata.obs["is_control"] = adata.obs.apply(
    lambda x: True if x["condition"] == "ctrl" else False, axis=1
)

In [ ]:
adata.obs[['perturbation_1', 'perturbation_2']] = adata.obs["perturbation_name"].str.split("+", expand=True)
# fill None with 'control'
adata.obs['perturbation_1'] = adata.obs['perturbation_1'].fillna('control')
adata.obs['perturbation_2'] = adata.obs['perturbation_2'].fillna('control')

In [ ]:
from mygene import MyGeneInfo

mg = MyGeneInfo()
genes = list(set(adata.obs["perturbation_1"]).union(set(adata.obs["perturbation_2"])))
out = mg.querymany(genes, scopes='symbol', fields='ensembl.gene', species='human')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


5 input query terms found no hit:	['ELMSAN1', 'C3orf72', 'KIAA1804', 'C19orf26', 'control']


In [ ]:
gene_name_map = {
    'control': 'control',
    'ELMSAN1': 'MIDEAS',
    'KIAA1804': 'MAP3K21',
    'C19orf26': 'CBARP',
    'C3orf72': 'FOXL2NB',
}
additional_genes = [gene_name_map[g] for g in gene_name_map.keys()]
out_2 = mg.querymany(additional_genes, scopes='symbol', fields='ensembl.gene', species='human')
out.extend(out_2)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found no hit:	['control']


In [ ]:
gene_name_old = list(gene_name_map.keys())
gene_to_ensembl = {}
for gene in out:
    if gene["query"] == 'control':
        gene_to_ensembl['control'] = 'control'
        continue
    if gene["query"] in additional_genes:
        old_gene_name = [k for k,v in gene_name_map.items() if v == gene["query"]][0]
    elif gene["query"] in gene_name_old:
        continue
    else:
        old_gene_name = gene["query"]
    try:
        if len(gene["ensembl"]) > 1:
            gene_to_ensembl[old_gene_name] = gene["ensembl"][0]["gene"] # take the first one
        else:
            gene_to_ensembl[old_gene_name] = gene["ensembl"]["gene"]
    except (KeyError, IndexError):
        print(f"Error processing gene: {gene}")

In [ ]:
gene_to_ensembl['control'] = 'control'

adata.obs["gene_ensembl_1"] = adata.obs["perturbation_1"].map(gene_to_ensembl)
adata.obs["gene_ensembl_2"] = adata.obs["perturbation_2"].map(gene_to_ensembl)

In [ ]:
# randomly select 5000 cells from controls
import numpy as np
adata_ctrl = adata[adata.obs["is_control"], :].copy()
np.random.seed(0)
selected_control_indices = np.random.choice(adata_ctrl.obs_names, size=5000, replace=False)
selected_control = adata_ctrl[selected_control_indices, :]

adata_no_ctrl = adata[~adata.obs["is_control"], :].copy()
adata = sc.concat([adata_no_ctrl, selected_control], axis=0)

In [ ]:
adata_path_esm = "../data/preprocessed_norman_combo_with_esm.h5ad"
get_esm_embedding_adata(adata, adata_path_esm)

# Training & Prediction

In [ ]:
%%bash
datasets=("replogle_k562" "replogle_rpe1" "nadig_hepg2" "nadig_jurkat")
adata_paths=(
    "../data/preprocessed_replogle_k562_with_esm.h5ad"
    "../data/preprocessed_replogle_rpe1_with_esm.h5ad"
    "../data/preprocessed_nadig_hepg2_with_esm.h5ad"
    "../data/preprocessed_nadig_jurkat_with_esm.h5ad"
)

# Define splits and fractions to iterate over
splits=(1 2 3)

# Iterate over all combinations
for i in "${!datasets[@]}"; do
    dataset="${datasets[$i]}"
    adata_path="${adata_paths[$i]}"
    
    for split in "${splits[@]}"; do
        split_path="../data/gears/${dataset}/splits/${dataset}_single_${split}_0.75.pkl"
        working_dir="../data/cellflow/${dataset}/"
        
        python ../scripts/cellflow_train.py ${adata_path} ${split_path} ${split} ${working_dir}
    done
done

In [ ]:
%%bash
dataset="norman_combo"
adata_path="../data/preprocessed_norman_combo_with_esm.h5ad"
splits=(1 2 3)

for split in "${splits[@]}"; do
    split_path="../data/gears/${dataset}/splits/${dataset}_single_${split}_0.75.pkl"
    working_dir="../data/cellflow/${dataset}/"
    
    python ../scripts/cellflow_train_norman_combo.py ${adata_path} ${split_path} ${split} ${working_dir}
done